<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/Disease_Prediction_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Disease Prediction using Machine Learning

# Project 8 — Bioinformatics

Is notebook mein hum symptoms ke basis par disease predict karain gay — patient jo symptoms select karta hai, unse 20 common diseases mein se most likely disease predict hoti hai, top differential diagnoses ke sath.

# **Pipeline:**
# 1. Symptom-Disease knowledge base banana
# 2. Synthetic patient records generate karna
# 3. Interactive Plotly visualizations (disease distribution, symptom-disease heatmap)
# 4. Multi-class classifier training
# 5. Evaluation (accuracy, confusion matrix, top predictive symptoms)
# 6. Runtime cell apne symptoms checkboxes se select karein, turant disease predict karein (top-3 differential diagnosis ke sath)


## 1. Setup & Imports

In [1]:
# !pip install -q plotly scikit-learn pandas numpy ipywidgets

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, top_k_accuracy_score

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)


## 2. Symptom–Disease Knowledge Base


In [2]:
disease_symptoms = {
    "Common Cold": ["runny_nose", "sneezing", "sore_throat", "cough", "mild_fever", "congestion"],
    "Influenza (Flu)": ["high_fever", "body_ache", "fatigue", "chills", "headache", "cough", "sore_throat"],
    "COVID-19": ["fever", "dry_cough", "fatigue", "loss_of_taste_smell", "shortness_of_breath", "body_ache"],
    "Pneumonia": ["fever", "cough_with_phlegm", "shortness_of_breath", "chest_pain", "fatigue", "chills"],
    "Asthma": ["shortness_of_breath", "wheezing", "chest_tightness", "cough", "difficulty_breathing"],
    "Migraine": ["headache", "nausea", "sensitivity_to_light", "sensitivity_to_sound", "visual_aura"],
    "Gastroenteritis": ["diarrhea", "vomiting", "abdominal_pain", "nausea", "mild_fever", "dehydration"],
    "Typhoid": ["prolonged_fever", "abdominal_pain", "weakness", "loss_of_appetite", "headache", "constipation"],
    "Dengue": ["high_fever", "severe_headache", "joint_pain", "muscle_pain", "rash", "eye_pain"],
    "Malaria": ["cyclic_fever", "chills", "sweating", "headache", "nausea", "muscle_pain"],
    "Urinary Tract Infection": ["burning_urination", "frequent_urination", "lower_abdominal_pain", "cloudy_urine", "mild_fever"],
    "Diabetes (Type 2)": ["frequent_urination", "excessive_thirst", "fatigue", "blurred_vision", "slow_healing_wounds"],
    "Hypertension": ["headache", "dizziness", "blurred_vision", "chest_pain", "shortness_of_breath"],
    "Anemia": ["fatigue", "pale_skin", "weakness", "dizziness", "shortness_of_breath", "cold_hands_feet"],
    "Allergic Rhinitis": ["sneezing", "runny_nose", "itchy_eyes", "congestion", "itchy_throat"],
    "Chickenpox": ["itchy_rash", "mild_fever", "fatigue", "loss_of_appetite", "headache"],
    "Tuberculosis": ["prolonged_cough", "weight_loss", "night_sweats", "fever", "fatigue", "chest_pain"],
    "Hepatitis A": ["jaundice", "fatigue", "nausea", "abdominal_pain", "loss_of_appetite", "dark_urine"],
    "Arthritis": ["joint_pain", "joint_stiffness", "swelling", "reduced_range_of_motion", "fatigue"],
    "Depression": ["persistent_sadness", "fatigue", "loss_of_interest", "sleep_problems", "difficulty_concentrating"],
}

all_symptoms = sorted(set(s for symptoms in disease_symptoms.values() for s in symptoms))
diseases = list(disease_symptoms.keys())

print(f"Total diseases: {len(diseases)}")
print(f"Total unique symptoms: {len(all_symptoms)}")


Total diseases: 20
Total unique symptoms: 64


## 3. Generate Synthetic Patient Dataset

In [3]:
def generate_patients(n_per_disease=60, symptom_prob=0.75, noise_prob=0.05, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    for disease, characteristic_symptoms in disease_symptoms.items():
        for _ in range(n_per_disease):
            row = {s: 0 for s in all_symptoms}
            # Patient shows each characteristic symptom with some probability (not always all)
            for s in characteristic_symptoms:
                row[s] = int(rng.random() < symptom_prob)
            # Small chance of a random unrelated symptom (noise)
            for s in all_symptoms:
                if s not in characteristic_symptoms and rng.random() < noise_prob:
                    row[s] = 1
            row["disease"] = disease
            rows.append(row)
    return pd.DataFrame(rows)

patients_df = generate_patients(n_per_disease=60)
print(f"Dataset shape: {patients_df.shape}")
patients_df.head()


Dataset shape: (1200, 65)


,abdominal_pain,blurred_vision,body_ache,burning_urination,chest_pain,chest_tightness,chills,cloudy_urine,cold_hands_feet,congestion,...,sneezing,sore_throat,sweating,swelling,visual_aura,vomiting,weakness,weight_loss,wheezing,disease
0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,Common Cold
1,0,0,0,0,0,0,0,0,0,1,...,1,1,0,0,0,1,0,0,0,Common Cold
2,0,1,0,0,0,0,0,0,0,1,...,1,1,0,0,0,0,0,0,0,Common Cold
3,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,Common Cold
4,0,0,0,0,0,0,0,1,0,0,...,1,1,0,0,0,0,0,0,0,Common Cold


## 4. Interactive Visualizations

In [4]:
disease_counts = patients_df["disease"].value_counts()
fig = px.bar(disease_counts, orientation='h', title="Patient Records per Disease (Class Balance)",
             labels={"value": "Count", "index": "Disease"}, template="plotly_white",
             color=disease_counts.values, color_continuous_scale="Tealgrn")
fig.update_layout(height=600, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig.show()


In [5]:
# Symptom-Disease association heatmap (mean presence rate)
heatmap_matrix = patients_df.groupby("disease")[all_symptoms].mean()

fig = px.imshow(
    heatmap_matrix.values, x=all_symptoms, y=heatmap_matrix.index,
    color_continuous_scale="Reds", aspect="auto",
    title="Symptom Presence Rate by Disease",
    labels=dict(color="Presence Rate")
)
fig.update_layout(height=700, xaxis_tickangle=-75)
fig.show()


In [6]:
symptom_freq = patients_df[all_symptoms].mean().sort_values(ascending=False).head(20)
fig = px.bar(symptom_freq, orientation='h', title="Top 20 Most Common Symptoms (Overall)",
             labels={"value": "Frequency", "index": "Symptom"}, template="plotly_white",
             color=symptom_freq.values, color_continuous_scale="Oranges")
fig.update_layout(height=600, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig.show()


## 5. Train Multi-Class Disease Classifier

In [7]:
le = LabelEncoder()
X = patients_df[all_symptoms]
y = le.fit_transform(patients_df["disease"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

disease_clf = RandomForestClassifier(n_estimators=400, random_state=42)
disease_clf.fit(X_train, y_train)

preds = disease_clf.predict(X_test)
probs = disease_clf.predict_proba(X_test)

acc = accuracy_score(y_test, preds)
top3_acc = top_k_accuracy_score(y_test, probs, k=3, labels=np.arange(len(le.classes_)))

print(f"Top-1 Accuracy: {acc:.3f}")
print(f"Top-3 Accuracy: {top3_acc:.3f}  (correct disease within top-3 predictions)")
print("\n", classification_report(y_test, preds, target_names=le.classes_, zero_division=0))


Top-1 Accuracy: 0.947
Top-3 Accuracy: 0.993  (correct disease within top-3 predictions)

                          precision    recall  f1-score   support

      Allergic Rhinitis       1.00      0.93      0.97        15
                 Anemia       0.88      1.00      0.94        15
              Arthritis       1.00      1.00      1.00        15
                 Asthma       1.00      0.93      0.97        15
               COVID-19       0.93      0.87      0.90        15
             Chickenpox       0.93      0.87      0.90        15
            Common Cold       0.83      1.00      0.91        15
                 Dengue       1.00      0.93      0.97        15
             Depression       1.00      0.87      0.93        15
      Diabetes (Type 2)       1.00      0.87      0.93        15
        Gastroenteritis       1.00      0.87      0.93        15
            Hepatitis A       0.88      0.93      0.90        15
           Hypertension       0.82      0.93      0.88        15

In [8]:
cm = confusion_matrix(y_test, preds)
fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues",
                 x=le.classes_, y=le.classes_,
                 labels=dict(x="Predicted", y="Actual", color="Count"),
                 title="Confusion Matrix — Disease Classifier")
fig.update_layout(height=700, xaxis_tickangle=-60)
fig.show()

importances = pd.Series(disease_clf.feature_importances_, index=all_symptoms).sort_values(ascending=False).head(20)
fig2 = px.bar(importances, orientation='h', title="Top 20 Most Predictive Symptoms (Global)",
              labels={"value": "Importance", "index": "Symptom"}, template="plotly_white",
              color=importances.values, color_continuous_scale="Viridis")
fig2.update_layout(height=600, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig2.show()


## 6.  Runtime Prediction — Apne Symptoms Select Karein




In [9]:
symptom_checkboxes = {}
checkbox_widgets = []
for s in all_symptoms:
    label = s.replace("_", " ").title()
    cb = widgets.Checkbox(value=False, description=label, indent=False,
                           layout=widgets.Layout(width='260px'))
    symptom_checkboxes[s] = cb
    checkbox_widgets.append(cb)

symptom_grid = widgets.GridBox(checkbox_widgets, layout=widgets.Layout(grid_template_columns="repeat(4, 270px)", grid_gap="4px"))

predict_btn = widgets.Button(description=" Disease Predict Karein", button_style='success',
                              layout=widgets.Layout(width='260px', height='42px'))
clear_btn = widgets.Button(description="↺ Clear Symptoms", layout=widgets.Layout(width='200px', height='42px'))
out = widgets.Output()

def render_disease_result(top_diseases, top_probs, selected_symptoms):
    primary = top_diseases[0]
    conf = top_probs[0] * 100

    rows_html = ""
    bar_colors = ["#E63946", "#F4A261", "#2E86AB"]
    for i, (d, p) in enumerate(zip(top_diseases, top_probs)):
        rows_html += f"""
        <div style="margin-top:8px;">
            <div style="display:flex; justify-content:space-between; font-size:14px;">
                <span>{i+1}. {d}</span><span><b>{p*100:.1f}%</b></span>
            </div>
            <div style="height:10px; width:100%; background:#e0e0e0; border-radius:5px; overflow:hidden; margin-top:3px;">
                <div style="height:100%; width:{p*100:.1f}%; background:{bar_colors[i]};"></div>
            </div>
        </div>
        """

    symptoms_html = ", ".join([s.replace("_", " ").title() for s in selected_symptoms]) if selected_symptoms else "Koi symptom select nahi kiya gaya"

    html = f"""
    <div style="border:2px solid #E63946; border-radius:12px; padding:18px; margin-top:12px; font-family:sans-serif; background:#fafafa;">
        <div style="font-size:22px; font-weight:700; color:#E63946;"> Most Likely: {primary}</div>
        <div style="font-size:14px; margin-top:4px; color:#333;">Confidence: <b>{conf:.1f}%</b></div>
        <div style="font-size:12px; color:#777; margin-top:6px;">Selected symptoms: {symptoms_html}</div>
        <hr style="margin:14px 0; border:none; border-top:1px solid #ddd;">
        <div style="font-size:14px; font-weight:600; color:#333;">Top 3 Differential Diagnosis:</div>
        {rows_html}
    </div>
    """
    display(HTML(html))

def on_predict(b):
    with out:
        clear_output()
        selected = [s for s, cb in symptom_checkboxes.items() if cb.value]
        if not selected:
            print(" Kam az kam ek symptom select karein.")
            return
        row = {s: (1 if s in selected else 0) for s in all_symptoms}
        row_df = pd.DataFrame([row])[all_symptoms]
        proba = disease_clf.predict_proba(row_df)[0]
        top_idx = proba.argsort()[::-1][:3]
        top_diseases = le.inverse_transform(top_idx)
        top_probs = proba[top_idx]
        render_disease_result(top_diseases, top_probs, selected)

def on_clear(b):
    for cb in symptom_checkboxes.values():
        cb.value = False
    with out:
        clear_output()

predict_btn.on_click(on_predict)
clear_btn.on_click(on_clear)

display(widgets.HTML("<b style='font-size:16px;'>Apne Symptoms Select Karein:</b>"))
display(symptom_grid)
display(widgets.HBox([predict_btn, clear_btn]))
display(out)


HTML(value="<b style='font-size:16px;'>Apne Symptoms Select Karein:</b>")

GridBox(children=(Checkbox(value=False, description='Abdominal Pain', indent=False, layout=Layout(width='260px…

Output()